# Estadísticas de la base de 340 sintonizaciones (experimento E5)

En esta libreta se describe la base de parámetros de la Arquitectura I sintonizados mediante algoritmo genético para los 340 pacientes virtuales (archivo `SINTONIZACIONES.csv`, generado en `PID_Filtro_EMA.ipynb`).

**Procedimiento.**
1. Distribución del RMSE por sexo y prueba U de Mann–Whitney.
2. Estadística descriptiva de los parámetros, coeficiente de variación y número de parámetros en los límites del intervalo de búsqueda.
3. Correlaciones de Spearman entre las covariables del paciente y los parámetros sintonizados.

En el archivo, la columna `gamma` corresponde al factor de escalamiento denotado como ρ en la tesis. El RMSE de cada sintonización se evalúa en las fases de sedación y mantenimiento, horizonte en el que actúa el controlador; durante la recuperación la planta evoluciona en lazo abierto. Los casos con sintonización deficiente se identifican como valores atípicos superiores del RMSE mediante el criterio de Tukey ($\mathrm{RMSE} > Q_3 + 1.5\,\mathrm{RIC}$), el mismo empleado en `KNN_Estimacion_PID.ipynb`.

In [1]:
# =============================================================================
# CONFIGURACIÓN Y CARGA DE DATOS
# =============================================================================
import numpy as np
import pandas as pd
from scipy import stats

RUTA_SINTONIZACIONES = "SINTONIZACIONES.csv"
PARAMETROS           = ["Kp", "Ki", "Kd", "gamma"]
COVARIABLES          = ["Edad", "Peso", "Altura"]

# Intervalo de búsqueda del experimento E5, parámetro a parámetro; coincide con
# GANANCIAS_INF/SUP y RHO_INF/SUP de PID_Filtro_EMA.ipynb
LIM_INF = {"Kp": 1e-9, "Ki": 1e-9, "Kd": 1e-9, "gamma": 1e-8}
LIM_SUP = {"Kp": 0.5,  "Ki": 0.5,  "Kd": 0.5,  "gamma": 3.0}
TOL_REL = 1e-3       # Tolerancia relativa para considerar un parámetro en el límite
FACTOR_TUKEY = 1.5   # Factor del rango intercuartílico del criterio de Tukey

s = pd.read_csv(RUTA_SINTONIZACIONES)
print(f"[INFO] Pacientes: {len(s)}   hombres: {(s.Sexo == 'M').sum()}   mujeres: {(s.Sexo == 'F').sum()}")

# Umbral de sintonización deficiente: valores atípicos superiores del RMSE
q1_rmse, q3_rmse = s["RMSE"].quantile([0.25, 0.75])
UMBRAL_RMSE = float(q3_rmse + FACTOR_TUKEY * (q3_rmse - q1_rmse))
print(f"[INFO] Umbral de sintonización deficiente (criterio de Tukey): "
      f"Q1 = {q1_rmse:.4f}   Q3 = {q3_rmse:.4f}   umbral = {UMBRAL_RMSE:.4f}")

[INFO] Pacientes: 340   hombres: 170   mujeres: 170
[INFO] Umbral de sintonización deficiente (criterio de Tukey): Q1 = 5.4289   Q3 = 5.7459   umbral = 6.2213


## Etapa 1: Desempeño por sexo

In [2]:
# =============================================================================
# RMSE POR SEXO Y CASOS CON SINTONIZACIÓN DEFICIENTE
# =============================================================================
print(s.groupby("Sexo")["RMSE"].describe().round(3))
u = stats.mannwhitneyu(s.loc[s.Sexo == "M", "RMSE"], s.loc[s.Sexo == "F", "RMSE"])
print(f"\nU de Mann-Whitney (hombres frente a mujeres): valor-p = {u.pvalue:.2e}")
print(f"RMSE total: {s.RMSE.mean():.2f} ± {s.RMSE.std():.2f}   "
      f"(mín. {s.RMSE.min():.2f}, máx. {s.RMSE.max():.2f})")
print(f"Casos con RMSE mayor que el umbral ({UMBRAL_RMSE:.4f}), por sexo:")
print((s.RMSE > UMBRAL_RMSE).groupby(s.Sexo).sum())

      count   mean    std    min    25%    50%    75%    max
Sexo                                                        
F     170.0  5.540  0.260  5.027  5.375  5.505  5.702  6.570
M     170.0  5.655  0.239  5.197  5.493  5.638  5.798  6.605

U de Mann-Whitney (hombres frente a mujeres): valor-p = 3.02e-06
RMSE total: 5.60 ± 0.26   (mín. 5.03, máx. 6.60)
Casos con RMSE mayor que el umbral (6.2213), por sexo:
Sexo
F    3
M    3
Name: RMSE, dtype: int64


## Etapa 2: Distribución de los parámetros sintonizados

In [3]:
# =============================================================================
# ESTADÍSTICA DESCRIPTIVA DE LOS PARÁMETROS
# =============================================================================
filas = []
for c in PARAMETROS:
    v = s[c]
    filas.append(dict(Parametro=c,
                      Media=v.mean(), Desv_est=v.std(), Mediana=v.median(),
                      Q1=v.quantile(0.25), Q3=v.quantile(0.75),
                      Coef_variacion=v.std() / v.mean(),
                      En_limite_inferior=int((v <= LIM_INF[c] * (1 + TOL_REL)).sum()),
                      En_limite_superior=int((v >= LIM_SUP[c] * (1 - TOL_REL)).sum())))
pd.DataFrame(filas).round(4)

,Parametro,Media,Desv_est,Mediana,Q1,Q3,Coef_variacion,En_limite_inferior,En_limite_superior
0,Kp,0.0048,0.0028,0.0043,0.0032,0.0056,0.5885,0,0
1,Ki,0.0000,0.0000,0.0000,0.0000,0.0000,0.1755,0,0
2,Kd,0.4243,0.1205,0.5000,0.3593,0.5000,0.2841,1,201
3,gamma,1.4419,0.6470,1.3715,0.9433,1.8882,0.4487,0,3


## Etapa 3: Relación entre covariables y parámetros

In [4]:
# =============================================================================
# CORRELACIONES DE SPEARMAN ENTRE COVARIABLES Y PARÁMETROS
# =============================================================================
filas = []
for c in PARAMETROS:
    for v in COVARIABLES + ["RMSE"]:
        r = stats.spearmanr(s[c], s[v])
        filas.append(dict(Parametro=c, Variable=v, rho_Spearman=r.statistic, Valor_p=r.pvalue))
correlaciones = pd.DataFrame(filas)
correlaciones.pivot(index="Variable", columns="Parametro", values="rho_Spearman").round(3)

Parametro,Kd,Ki,Kp,gamma
Variable,,,,
Altura,-0.041,-0.014,0.039,0.002
Edad,-0.050,-0.186,-0.171,0.082
Peso,-0.054,-0.091,-0.003,-0.029
RMSE,-0.312,0.180,0.531,-0.330
